# WINPACT Unified Simulation API

## Two–Step Workflow: Build → Run

The unified experiment interface provides a single entry point to run both **single simulations** and **parameter sweeps** using the same function:

```python
from core.Simulation import build_experiment
```

The workflow is always:

1. **Build** an experiment object
2. **Run** it

---

## 1. `build_experiment(...)`

```python
exp = build_experiment(
    library_path,
    base_config_path,
    simulation_config,
    *,
    overrides=None,
    seed=None,
    parameter_space=None,
    base_seed=0,
    replicates=1,
    max_runs=None,
    zip_groups=None,
    name=None,
    result_directory=None,
    debug=True,
)
```

Creates a configured experiment object (`SingleExperiment` or `SweepExperiment`)
depending on the inputs.

---

## Required Arguments

### `library_path`

**Type:** `str | Path`
Path to the input-library folder that contains the YAML config structure (e.g. `examples/Inputs/HKN`).

---

### `base_config_path`

**Type:** `str | Path`
Relative or absolute path to the base YAML configuration file
(e.g. `"config.yaml"`).

---

### `simulation_config`

**Type:** `SimulationConfig | dict`
A mapping defining which modules of the simulation should run.

Example:

```python
sim_cfg = {
    "run_capex": True,
    "capex_dashboard": False,
    "run_windfarm": True,
    "run_opex": True,
    ...
}
```

This is mandatory because it controls the module-level execution behavior inside `ValueWindEnv`.

---

## Optional Arguments (Single-Run Features)

### `overrides`

**Type:** `dict | None`
Dotted-path overrides that modify fields inside the base YAML configuration.

Example:

```python
overrides={
    "CAPEX_overrides.material_data.Commodity.Steel.CostParameters.material_cost": 1400
}
```

Use this to vary a single simulation without defining a full parameter sweep.

---

### `seed`

**Type:** `int | None`
An explicit random seed for deterministic single simulations.

If omitted, `base_seed` (default=0) is used to generate a seed.

---

## Optional Arguments (Sweep & Replicates)

### `parameter_space`

**Type:** `dict[str, list] | None`
If provided, the experiment becomes a **parameter sweep**.

Example:

```python
parameter_space={
    "CAPEX_overrides.material_data.Commodity.Steel.CostParameters.material_cost": [800, 1200, 1600],
    "Revenue_overrides.market_price_scenario": ["low", "mid", "high"],
}
```

---

### `replicates`

**Type:** `int` (default: `1`)
Number of **stochastic realisations** for each scenario.

* When `parameter_space is None`:
  → Builds repeated runs of one configuration.
* When `parameter_space is provided`:
  → Multiplies each parameter combination by `replicates`.

---

### `base_seed`

**Type:** `int` (default: `0`)
Used to generate deterministic but distinct seeds for each replicate & scenario.

Ignored if explicit `seed` is given for a single-run setup.

---

### `max_runs`

**Type:** `int | None`
Caps the total number of generated scenarios, useful for debugging.

---

### `zip_groups`

**Type:** `dict[str, list[str]] | None`
Allows grouped parameters to be zipped together positionally
instead of forming a Cartesian product.

---

## Optional Metadata

### `name`

**Type:** `str | None`
The experiment name to embed into each scenario.

---

### `result_directory`

**Type:** `str | Path | None`
Folder where scenario definitions and run indexes are saved.

If omitted, defaults to `"results"`.

---

### `debug`

**Type:** `bool`, default: `True`
If `True`, exceptions propagate.
If `False`, failing scenarios are caught and recorded.

---

## 2. Running the Experiment

```python
results_df = exp.run()
```

* Executes **all** simulations (single or sweep).
* Returns a `pandas.DataFrame` where each row represents a scenario run:

| column             | description                            |
| ------------------ | -------------------------------------- |
| `scenario_id`      | unique hash for the scenario           |
| `label`            | human-readable scenario label          |
| `seed`             | seed used for the run                  |
| `status`           | `"success"` or `"failed"`              |
| `duration_s`       | runtime duration                       |
| `error_message`    | captured exception message (if failed) |
| `traceback`        | full traceback (if debug=False)        |
| `experiment_name`  | name set in the builder                |
| `result_directory` | output directory                       |



# Single Experiment

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent   # parent of examples = WINPACT_TREND
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"

sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": True,
    "run_windfarm": True,
    "run_opex": True,
    "opex_dashboard": True,
    "run_lifetime_extension": True,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": True,
    "collect_results": True,
}

# build the experiment

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="config.yaml",
    simulation_config=sim_cfg,
    replicates = 1, 
    seed = 42,
    name="test_experiment",
    result_directory= str(RESULT_DIR),
)

# run the experiment
exp.run()


m:\Projects\Cost Model\HiperSim\WINPACT\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)
c:\Users\mograf\AppData\Local\anaconda3\envs\WFLoads311\Lib\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




[prices] first=2017-03-31 00:00:00  last=2041-11-24 15:00:00.000000001  count=216113
[power ] first=2017-03-31 00:00:00  last=2041-11-24 15:00:00.000000001  count=216113
Timestamps identical (length & order): True


m:\Projects\Cost Model\HiperSim\WINPACT\core\Revenue_Model.py:281: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,scenario_id,label,seed,status,duration_s,error_message,traceback,experiment_name,result_directory
0,c23d2d71e9,baseline,1073859781,success,3.75886,None,None,test_experiment,m:\Projects\Cost Model\HiperSim\WINPACT\results


# WINPACT Call with Scenario Designer

In [ ]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent   
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"

# Define the simulation configuration

sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": False,
    "run_windfarm": True,
    "run_opex": True,
    "opex_dashboard": False,
    "run_lifetime_extension": False,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": False,
    "collect_results": True,
}


# Define the Design of Experiments
parameter_space = {
    "Revenue_overrides.strike_price": [
        80, 100
    ],
    "Revenue_overrides.scheme_type": [
        "FiT", "CfD"
    ],
    "Scenario.name": [
        "FiT",
        "CfD",
    ],
}

zip_groups = {
    "macro_scenarios": [
        "Revenue_overrides.strike_price",
        "Revenue_overrides.scheme_type",
        "Scenario.name",
    ]
}



# build the experiment
exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",
    parameter_space=parameter_space,
    simulation_config = sim_cfg,
    base_seed=42,
    replicates=1,
    name="DoE_Experiment",
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,
)

# run the experiment
exp.run()


m:\Projects\Cost Model\HiperSim\WINPACT\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Cost Model\HiperSim\WINPACT\core\Revenue_Model.py:281: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

m:\Projects\Cost Model\HiperSim\WINPACT\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Cost Model\HiperSim\WINPACT\core\Revenue_Model.py:281: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,scenario_id,label,seed,status,duration_s,error_message,traceback,experiment_name,result_directory
0,30ab7c23b0,scheme_type=FiT__strike_price=80__name=FiT,574835255,success,1.682173,None,None,HKN_TI_boost_test,m:\Projects\Cost Model\HiperSim\WINPACT\results
1,72cb28d0f6,scheme_type=CfD__strike_price=100__name=CfD,437439553,success,1.778584,None,None,HKN_TI_boost_test,m:\Projects\Cost Model\HiperSim\WINPACT\results



# Parallel execution for HPC

This section documents how to enable **parallel execution** for WINPACT experiments while keeping the **default sequential behavior** unchanged.

---

## Overview

`build_experiment(...)` supports an optional `execution` argument that controls how scenario runs are dispatched.

* **Default behavior** (no `execution` argument):

  * Runs **sequentially**
  * Works on laptops, login nodes, and HPC systems

* **Parallel behavior** (opt-in):

  * Runs multiple scenarios concurrently
  * Uses Python executors (`process` or `thread` backends)

Sequential execution is always the safe default.

---

## Execution configuration

The `execution` argument can be provided as a **dictionary** (recommended for notebooks) or as an `ExecutionConfig` object.

### Dictionary form

```python
execution = {
    "backend": "sequential" | "process" | "thread",
    "n_jobs": 1,
    "chunksize": 1,  # optional
}
```

### Backends

| Backend      | When to use                                    |
| ------------ | ---------------------------------------------- |
| `sequential` | Default, debugging, safest option              |
| `process`    | CPU-bound workloads, best scaling              |
| `thread`     | I/O-bound work or when pickling is problematic |

Notes:

* `n_jobs <= 1` always results in sequential execution.
* `chunksize` is reserved for future tuning and may be ignored.

---

## Important constraints

### Debug mode

Parallel execution **requires**:

```python
debug=False
```

`debug=True` is intentionally disallowed with parallel backends because debug mode re-raises exceptions immediately, which conflicts with executor-based scheduling.

---

### Output safety

When running in parallel:

* Ensure each scenario writes to a **unique output directory or filename**
* Avoid shared fixed filenames across scenarios

Best practice:

* Include `scenario_id` or `scenario_label` in output paths

---

## Choosing `n_jobs`

### Local machine

```python
import os
n_jobs = os.cpu_count() or 1
```

### SLURM / HPC systems

Use the number of CPUs allocated to the job:

```python
import os
n_jobs = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
```

This ensures WINPACT respects scheduler allocations.

---

## Example 1 — Sequential execution (default)

No changes required. This is how WINPACT behaves by default:

```python
exp = build_experiment(
    library_path=LIBRARY_PATH,
    base_config_path="Config.yaml",
    simulation_config=sim_cfg,
    parameter_space=parameter_space,
    name="example_sequential",
    result_directory=RESULT_DIR,
)

results = exp.run()
```

---

## Example 2 — Explicit sequential execution

Equivalent to the default, but explicit:

```python
exp = build_experiment(
    library_path=LIBRARY_PATH,
    base_config_path="Config.yaml",
    simulation_config=sim_cfg,
    parameter_space=parameter_space,
    name="example_sequential_explicit",
    result_directory=RESULT_DIR,
    execution={"backend": "sequential"},
)

results = exp.run()
```

---

## Example 3 — Parallel execution (process backend)

This example enables parallel execution using multiple CPU cores.

```python
import os

n_jobs = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
if n_jobs <= 0:
    n_jobs = 1

exp = build_experiment(
    library_path=LIBRARY_PATH,
    base_config_path="Config.yaml",
    simulation_config=sim_cfg,
    parameter_space=parameter_space,
    base_seed=42,
    replicates=1,
    name="example_parallel",
    result_directory=RESULT_DIR,
    zip_groups=zip_groups,

    # Enable parallel execution
    execution={"backend": "process", "n_jobs": n_jobs},

    # Required for parallel execution
    debug=False,
)

results = exp.run()
```

---

## Switching to threads

If process-based execution fails due to pickling constraints, use threads:

```python
execution={"backend": "thread", "n_jobs": n_jobs}
```

This avoids pickling but may scale less for CPU-bound workloads.

---

## Summary

* Sequential execution is the default and safest option
* Parallel execution is **opt-in** and requires `debug=False`
* Use `process` for CPU-heavy workloads, `thread` for I/O-heavy or restricted environments
* Always ensure output paths are scenario-safe when running in parallel


In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"

sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": False,
    "run_windfarm": True,
    "run_opex": True,
    "opex_dashboard": False,
    "run_lifetime_extension": False,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": False,
    "collect_results": True,
}

# Define the Design of Experiments
parameter_space = {
    "Revenue_overrides.strike_price": [
        80, 100
    ],
    "Revenue_overrides.scheme_type": [
        "FiT", "CfD"
    ],
    "Scenario.name": [
        "FiT",
        "CfD",
    ],
}

zip_groups = {
    "macro_scenarios": [
        "Revenue_overrides.strike_price",
        "Revenue_overrides.scheme_type",
        "Scenario.name",
    ]
}

# Choose n_jobs from SLURM if available; otherwise fall back to local CPU count (or 1)
n_jobs = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
if n_jobs <= 0:
    n_jobs = 1

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",
    parameter_space=parameter_space,
    simulation_config=sim_cfg,
    base_seed=42,
    replicates=1,
    name="HKN_TI_boost_parallel",
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,

    # ✅ NEW: parallel execution controls
    execution={"backend": "process", "n_jobs": n_jobs},

    # ✅ REQUIRED for parallel execution
    debug=False,
)

exp.run()

